The implementation of **DDPG (Deep Deterministic Policy Gradient)** algorithm.

Article Name: **Continuous Control with Deep Reinforcement Learning**

link: https://arxiv.org/abs/1509.02971

In [6]:
# Environment Setup & Core Dependencies
!pip install -q "gymnasium[box2d,mujoco]"
!pip install -q tensorboard matplotlib

import os
import time
import random
import numpy as np
import torch as T
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter
import gymnasium as gym
import matplotlib.pyplot as plt

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    T.manual_seed(seed)
    T.cuda.manual_seed(seed)
    T.cuda.manual_seed_all(seed)
    T.backends.cudnn.deterministic = True

device = T.device('cuda:0' if T.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda:0


In [7]:
# Replay Buffer, Noise Generator, Network Architecture & TD3 Agent

class ReplayBuffer:
    def __init__(self, max_size, input_shape, n_actions):
        self.mem_size = max_size
        self.mem_cntr = 0
        self.state_memory = np.zeros((self.mem_size, *input_shape), dtype=np.float32)
        self.new_state_memory = np.zeros((self.mem_size, *input_shape), dtype=np.float32)
        self.action_memory = np.zeros((self.mem_size, n_actions), dtype=np.float32)
        self.reward_memory = np.zeros(self.mem_size, dtype=np.float32)
        self.terminal_memory = np.zeros(self.mem_size, dtype=np.bool_)

    def store_transition(self, state, action, reward, state_, done):
        index = self.mem_cntr % self.mem_size
        self.state_memory[index] = state
        self.new_state_memory[index] = state_
        self.action_memory[index] = action
        self.reward_memory[index] = reward
        self.terminal_memory[index] = done
        self.mem_cntr += 1

    def sample_buffer(self, batch_size):
        max_mem = min(self.mem_cntr, self.mem_size)
        batch = np.random.choice(max_mem, batch_size, replace=False)

        states = self.state_memory[batch]
        actions = self.action_memory[batch]
        rewards = self.reward_memory[batch]
        states_ = self.new_state_memory[batch]
        dones = self.terminal_memory[batch]

        return states, actions, rewards, states_, dones


class OUActionNoise:
    def __init__(self, mean, std_deviation, theta=0.15, dt=1e-2, x_initial=None):
        self.theta = theta
        self.mean = mean
        self.std_dev = std_deviation
        self.dt = dt
        self.x_initial = x_initial
        self.reset()

    def __call__(self):
        x = (
            self.x_prev
            + self.theta * (self.mean - self.x_prev) * self.dt
            + self.std_dev * np.sqrt(self.dt) * np.random.normal(size=self.mean.shape)
        )
        self.x_prev = x
        return x

    def reset(self):
        self.x_prev = self.x_initial if self.x_initial is not None else np.zeros_like(self.mean)


class ActorNetwork(nn.Module):
    def __init__(self, alpha, input_dims, fc1_dims, fc2_dims, n_actions, max_action):
        super(ActorNetwork, self).__init__()
        self.max_action = max_action
        self.fc1 = nn.Linear(*input_dims, fc1_dims)
        self.fc2 = nn.Linear(fc1_dims, fc2_dims)
        self.mu = nn.Linear(fc2_dims, n_actions)

        self.optimizer = optim.Adam(self.parameters(), lr=alpha)

    def forward(self, state):
        prob = F.relu(self.fc1(state))
        prob = F.relu(self.fc2(prob))
        mu = T.tanh(self.mu(prob)) * self.max_action
        return mu


class CriticNetwork(nn.Module):
    def __init__(self, beta, input_dims, fc1_dims, fc2_dims, n_actions):
        super(CriticNetwork, self).__init__()
        # Twin Critics Architecture to mitigate Overestimation Bias
        self.q1_fc1 = nn.Linear(input_dims[0] + n_actions, fc1_dims)
        self.q1_fc2 = nn.Linear(fc1_dims, fc2_dims)
        self.q1_out = nn.Linear(fc2_dims, 1)

        self.q2_fc1 = nn.Linear(input_dims[0] + n_actions, fc1_dims)
        self.q2_fc2 = nn.Linear(fc1_dims, fc2_dims)
        self.q2_out = nn.Linear(fc2_dims, 1)

        self.optimizer = optim.Adam(self.parameters(), lr=beta)

    def forward(self, state, action):
        sa = T.cat([state, action], dim=1)

        q1 = F.relu(self.q1_fc1(sa))
        q1 = F.relu(self.q1_fc2(q1))
        q1 = self.q1_out(q1)

        q2 = F.relu(self.q2_fc1(sa))
        q2 = F.relu(self.q2_fc2(q2))
        q2 = self.q2_out(q2)

        return q1, q2

    def Q1(self, state, action):
        sa = T.cat([state, action], dim=1)
        q1 = F.relu(self.q1_fc1(sa))
        q1 = F.relu(self.q1_fc2(q1))
        return self.q1_out(q1)


class TD3Agent:
    def __init__(self, alpha, beta, input_dims, tau, env, gamma=0.99,
                 update_actor_interval=2, warmup=1000, n_actions=2, max_size=1000000,
                 layer1_size=400, layer2_size=300, batch_size=100, noise=0.1):
        self.gamma = gamma
        self.tau = tau
        self.max_action = float(env.action_space.high[0])
        self.min_action = float(env.action_space.low[0])
        self.memory = ReplayBuffer(max_size, input_dims, n_actions)
        self.batch_size = batch_size
        self.n_actions = n_actions
        self.learn_step_cntr = 0
        self.time_step = 0
        self.warmup = warmup
        self.update_actor_iter = update_actor_interval

        self.actor = ActorNetwork(alpha, input_dims, layer1_size, layer2_size, n_actions, self.max_action).to(device)
        self.critic = CriticNetwork(beta, input_dims, layer1_size, layer2_size, n_actions).to(device)

        self.target_actor = ActorNetwork(alpha, input_dims, layer1_size, layer2_size, n_actions, self.max_action).to(device)
        self.target_critic = CriticNetwork(beta, input_dims, layer1_size, layer2_size, n_actions).to(device)

        self.noise = noise
        self.noise_generator = OUActionNoise(mean=np.zeros(n_actions), std_deviation=float(noise) * np.ones(n_actions))

        self.update_network_parameters(tau=1.0)

    def choose_action(self, observation, evaluate=False):
        if self.time_step < self.warmup and not evaluate:
            action = np.random.uniform(self.min_action, self.max_action, size=self.n_actions)
        else:
            state = T.tensor(np.array([observation]), dtype=T.float32).to(device)
            self.actor.eval()
            with T.no_grad():
                action = self.actor(state).cpu().numpy()[0]
            self.actor.train()

            if not evaluate:
                action += self.noise_generator()

        action = np.clip(action, self.min_action, self.max_action)
        self.time_step += 1
        return action

    def remember(self, state, action, reward, new_state, done):
        self.memory.store_transition(state, action, reward, new_state, done)

    def learn(self, writer=None, global_step=0):
        if self.memory.mem_cntr < self.batch_size:
            return

        state, action, reward, new_state, done = self.memory.sample_buffer(self.batch_size)

        states = T.tensor(state, dtype=T.float32).to(device)
        actions = T.tensor(action, dtype=T.float32).to(device)
        rewards = T.tensor(reward, dtype=T.float32).unsqueeze(1).to(device)
        states_ = T.tensor(new_state, dtype=T.float32).to(device)
        dones = T.tensor(done, dtype=T.bool).unsqueeze(1).to(device)

        with T.no_grad():
            # Target Policy Smoothing Noise
            target_actions = self.target_actor(states_)
            target_noise = T.clamp(T.randn_like(target_actions) * 0.2, -0.5, 0.5)
            target_actions = T.clamp(target_actions + target_noise, self.min_action, self.max_action)

            q1_target, q2_target = self.target_critic(states_, target_actions)
            q_target = T.min(q1_target, q2_target)
            q_target[dones] = 0.0
            y = rewards + self.gamma * q_target

        q1, q2 = self.critic(states, actions)
        critic_loss = F.mse_loss(q1, y) + F.mse_loss(q2, y)

        self.critic.optimizer.zero_grad()
        critic_loss.backward()
        self.critic.optimizer.step()

        self.learn_step_cntr += 1

        # Delayed Policy Update
        if self.learn_step_cntr % self.update_actor_iter == 0:
            actor_loss = -self.critic.Q1(states, self.actor(states)).mean()

            self.actor.optimizer.zero_grad()
            actor_loss.backward()
            self.actor.optimizer.step()

            self.update_network_parameters()

            if writer is not None:
                writer.add_scalar('Loss/Actor', actor_loss.item(), global_step)
                writer.add_scalar('Loss/Critic', critic_loss.item(), global_step)

    def update_network_parameters(self, tau=None):
        if tau is None:
            tau = self.tau

        for actor_param, target_actor_param in zip(self.actor.parameters(), self.target_actor.parameters()):
            target_actor_param.data.copy_(tau * actor_param.data + (1.0 - tau) * target_actor_param.data)

        for critic_param, target_critic_param in zip(self.critic.parameters(), self.target_critic.parameters()):
            target_critic_param.data.copy_(tau * critic_param.data + (1.0 - tau) * target_critic_param.data)

In [8]:
# Multi-Seed Experiment Engine

def train_seed(env_id, seed, n_games=200):
    set_seed(seed)
    env = gym.make(env_id)

    writer = SummaryWriter(f'runs/TD3_{env_id}_seed_{seed}')
    agent = TD3Agent(
        alpha=0.001,
        beta=0.001,
        input_dims=env.observation_space.shape,
        tau=0.005,
        env=env,
        batch_size=100,
        n_actions=env.action_space.shape[0],
        warmup=1000
    )

    score_history = []
    global_step = 0

    print(f"--- Training on {env_id} | Seed {seed} ---")

    for i in range(n_games):
        obs, _ = env.reset(seed=seed + i)
        done = False
        score = 0

        while not done:
            action = agent.choose_action(obs)
            obs_, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            agent.remember(obs, action, reward, obs_, done)
            agent.learn(writer, global_step)

            score += reward
            obs = obs_
            global_step += 1

        score_history.append(score)
        avg_score = np.mean(score_history[-100:])
        writer.add_scalar('Reward/Episode_Score', score, i)
        writer.add_scalar('Reward/Average_100_Episodes', avg_score, i)

        if (i + 1) % 50 == 0:
            print(f"Episode {i+1:03d}/{n_games} | Score: {score:.1f} | 100-Ep Avg: {avg_score:.1f}")

    env.close()
    writer.close()
    return score_history

In [9]:
# Execution, Statistical Evaluation & Publication-Ready Plotting

ENV_NAME = 'LunarLanderContinuous-v3'  # Alternative option: 'BipedalWalker-v3' or 'Pendulum-v1'
SEEDS = [10, 20, 30]
N_GAMES = 200

all_seed_scores = []

for seed in SEEDS:
    scores = train_seed(ENV_NAME, seed=seed, n_games=N_GAMES)
    all_seed_scores.append(scores)

all_seed_scores = np.array(all_seed_scores)

# Calculate Mean and Standard Deviation across Seeds
mean_scores = np.mean(all_seed_scores, axis=0)
std_scores = np.std(all_seed_scores, axis=0)

# Smooth curves for clean visualization
def smooth_signal(data, window_size=10):
    kernel = np.ones(window_size) / window_size
    return np.convolve(data, kernel, mode='same')

smoothed_mean = smooth_signal(mean_scores)
smoothed_std = smooth_signal(std_scores)

episodes = np.arange(1, N_GAMES + 1)

plt.figure(figsize=(10, 6))
plt.plot(episodes, smoothed_mean, label='TD3 Mean Reward', color='#1f77b4', linewidth=2)
plt.fill_between(
    episodes,
    smoothed_mean - smoothed_std,
    smoothed_mean + smoothed_std,
    color='#1f77b4',
    alpha=0.25,
    label='Std Dev (Reproducibility)'
)

plt.title(f'Performance Benchmark: TD3 on {ENV_NAME}', fontsize=14, fontweight='bold')
plt.xlabel('Episodes', fontsize=12)
plt.ylabel('Cumulative Reward', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='upper left', fontsize=11)
plt.tight_layout()
plt.savefig('td3_benchmark_results.png', dpi=300)
plt.show()

--- Training on LunarLanderContinuous-v3 | Seed 10 ---
Episode 050/200 | Score: -40.7 | 100-Ep Avg: -168.1
Episode 100/200 | Score: 42.7 | 100-Ep Avg: -97.0


KeyboardInterrupt: 